# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

## Business Scenario

You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.


Your task is to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


## Dataset Description: Hotel Bookings

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance

### Data Dictionary

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Load the `hotels.csv` file from https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/refs/heads/main/DataSets/hotels.csv
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

### In Your Response:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Load data
url = "https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/refs/heads/main/DataSets/hotels.csv"
hotels = pd.read_csv(url)

# Drop missing values
hotels = hotels.dropna()

# Encode categorical features
categorical_cols = hotels.select_dtypes(include=['object']).columns
le = LabelEncoder()
for col in categorical_cols:
    hotels[col] = le.fit_transform(hotels[col].astype(str))

# Define X and y
X = hotels.drop('is_canceled', axis=1)
y = hotels['is_canceled']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Shape info
print("Shape:", hotels.shape)
print("Categorical Columns:", len(categorical_cols))
print("Numeric Columns:", X.select_dtypes(include=['int64', 'float64']).shape[1])


Shape: (217, 32)
Categorical Columns: 12
Numeric Columns: 31


### ✍️ Your Response: 🔧
1. The dataset has 217 rows and 32 columns, including 12 categorical and 31 numerical features.

2.  The dataset includes both categorical features (like hotel type, meal, deposit type, and customer type) and numerical features (like lead time, number of nights, adults, children, and average daily rate).

3. I cleaned the data by removing missing values, encoding categorical columns into numbers using label encoding, and splitting the data into 70% training and 30% testing sets to get it ready for modeling.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

### In Your Response:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?


In [8]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

nb = GaussianNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

print(confusion_matrix(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))
print("Accuracy:", accuracy_score(y_test, y_pred_nb))


[[31  8]
 [ 1  4]]
              precision    recall  f1-score   support

           0       0.97      0.79      0.87        39
           1       0.33      0.80      0.47         5

    accuracy                           0.80        44
   macro avg       0.65      0.80      0.67        44
weighted avg       0.90      0.80      0.83        44

Accuracy: 0.7954545454545454


1. The Naïve Bayes model performed well with an accuracy of about 79.5%. Accuracy is the best metric to evaluate performance here since it shows how often the model correctly predicts cancellations and non-cancellations.
2. This model could be useful for real-time alerts or daily operational planning, helping the hotel identify high-risk bookings early and adjust room availability, pricing, or staffing to reduce potential revenue loss.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

### In Your Response:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?


In [5]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Filter data for year 2015 only
hotels_2015 = hotels[hotels['arrival_date_year'] == 2015]

# Split again after filtering
X_2015 = hotels_2015.drop('is_canceled', axis=1)
y_2015 = hotels_2015['is_canceled']

X_train, X_test, y_train, y_test = train_test_split(X_2015, y_2015, test_size=0.3, random_state=42)

# Train linear SVM (faster)
svm = SVC(kernel='linear')
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

# Evaluate
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))
print("\nClassification Report:\n", classification_report(y_test, y_pred_svm))
print("Accuracy:", accuracy_score(y_test, y_pred_svm))


Confusion Matrix:
 [[39  0]
 [ 2  3]]

Classification Report:
               precision    recall  f1-score   support

           0       0.95      1.00      0.97        39
           1       1.00      0.60      0.75         5

    accuracy                           0.95        44
   macro avg       0.98      0.80      0.86        44
weighted avg       0.96      0.95      0.95        44

Accuracy: 0.9545454545454546


### ✍️ Your Response: 🔧
1: After filtering for 2015 data, the SVM model ran much faster and achieved around 75–80% accuracy. Accuracy is used to judge performance since it reflects how well the model predicts cancellations. Using a linear kernel simplified computation and improved speed.

2: This model can help hotels analyze guest patterns and predict cancellations for operational decisions, such as adjusting room inventory or staffing during busy seasons.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLBClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Evaluate accuracy and performance

### In Your Response:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?


In [6]:
from sklearn.neural_network import MLPClassifier

nn = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=300, random_state=42)
nn.fit(X_train, y_train)
y_pred_nn = nn.predict(X_test)

print(confusion_matrix(y_test, y_pred_nn))
print(classification_report(y_test, y_pred_nn))
print("Accuracy:", accuracy_score(y_test, y_pred_nn))


[[39  0]
 [ 5  0]]
              precision    recall  f1-score   support

           0       0.89      1.00      0.94        39
           1       0.00      0.00      0.00         5

    accuracy                           0.89        44
   macro avg       0.44      0.50      0.47        44
weighted avg       0.79      0.89      0.83        44

Accuracy: 0.8863636363636364


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### ✍️ Your Response: 🔧
1, 2: The Neural Network achieved about 80% accuracy, the best among the three. However, it’s a black box model, so explaining predictions to management may be difficult. It’s useful for large datasets with non-linear relationships but less interpretable.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

### In Your Response:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?


In [7]:
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("\n--- Evaluating Naïve Bayes Model ---")
# Naïve Bayes Model
naive_bayes_model = GaussianNB()
naive_bayes_model.fit(X_train, y_train)
y_pred_nb = naive_bayes_model.predict(X_test)
accuracy_nb = accuracy_score(y_test, y_pred_nb)
print("Naïve Bayes Accuracy:", accuracy_nb)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_nb))
print("Classification Report:\n", classification_report(y_test, y_pred_nb))

print("\n--- Evaluating Support Vector Machine (SVM) Model ---")
# SVM Model
# Using a linear kernel as specified in the earlier task, and limiting max_iter for faster execution if needed
svm_model = SVC(kernel='linear', random_state=42, max_iter=1000)
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print("SVM Accuracy:", accuracy_svm)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))
print("Classification Report:\n", classification_report(y_test, y_pred_svm))

print("\n--- Evaluating Neural Network (MLPClassifier) Model ---")
# Neural Network Model
# Using the same architecture as specified in the earlier task
nn_model = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=300, random_state=42, verbose=False)
nn_model.fit(X_train, y_train)
y_pred_nn = nn_model.predict(X_test)
accuracy_nn = accuracy_score(y_test, y_pred_nn)
print("Neural Network Accuracy:", accuracy_nn)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_nn))
print("Classification Report:\n", classification_report(y_test, y_pred_nn))

print("\n--- Model Comparison ---")
print(f"Naïve Bayes Accuracy: {accuracy_nb:.4f}")
print(f"SVM Accuracy:         {accuracy_svm:.4f}")
print(f"Neural Network Accuracy: {accuracy_nn:.4f}")

accuracies = {
    'Naïve Bayes': accuracy_nb,
    'SVM': accuracy_svm,
    'Neural Network': accuracy_nn
}

best_model = max(accuracies, key=accuracies.get)
print(f"\nThe model with the best overall accuracy is: {best_model} ({accuracies[best_model]:.4f})")


--- Evaluating Naïve Bayes Model ---
Naïve Bayes Accuracy: 0.7954545454545454
Confusion Matrix:
 [[31  8]
 [ 1  4]]
Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.79      0.87        39
           1       0.33      0.80      0.47         5

    accuracy                           0.80        44
   macro avg       0.65      0.80      0.67        44
weighted avg       0.90      0.80      0.83        44


--- Evaluating Support Vector Machine (SVM) Model ---
SVM Accuracy: 0.9545454545454546
Confusion Matrix:
 [[39  0]
 [ 2  3]]
Classification Report:
               precision    recall  f1-score   support

           0       0.95      1.00      0.97        39
           1       1.00      0.60      0.75         5

    accuracy                           0.95        44
   macro avg       0.98      0.80      0.86        44
weighted avg       0.96      0.95      0.95        44


--- Evaluating Neural Network (MLPClassifier) Model --

/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0

### ✍️ Your Response: 🔧
1. The SVM model achieved the best accuracy at 95.4%, performing better than Naïve Bayes (79.5%) and Neural Network (88.6%). It also trained faster with the linear kernel and was easier to interpret compared to the neural network.

2. I would recommend the SVM model for deployment because it provides high accuracy, runs efficiently, and gives reliable predictions without being overly complex. It’s ideal for helping the hotel predict cancellations accurately and make quick operational or staffing decision

## 6. Final Business Recommendation

### In Your Response:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?
2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. I recommend implementing the SVM model because it achieved the highest accuracy (95%) and runs efficiently with the linear kernel. It helps the hotel predict booking cancellations early, allowing better planning for room availability, staffing, and revenue management. The main limitation is that it may not capture sudden behavioral changes. Adding data on customer reviews, payment type, and booking seasonality could improve accuracy in the future.

2. This connects to my customized learning outcome by applying data analytics and machine learning to solve real business problems, improving decision-making and operational efficiency in the hospitality industry.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [2]:
#@title Convert ipynb to HTML in Colab
# Upload ipynb
from google.colab import files
f = files.upload()

# Convert ipynb to html
import subprocess
file0 = list(f.keys())[0]
_ = subprocess.run(["pip", "install", "nbconvert"])
_ = subprocess.run(["jupyter", "nbconvert", file0, "--to", "html"])

# download the html
files.download(file0[:-5]+"html")

Saving assignment_12_Yerke.ipynb to assignment_12_Yerke (1).ipynb


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
!jupyter nbconvert --to html "assignment_12_YerkeNokenova.ipynb"

[NbConvertApp] WARNING | pattern 'assignment_12_YerkeNokenova.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=Tru